# Chapter 02: Generating Groups

**Source span:** printed pp. 9-50; physical PDF pp. 20-61. The local PDF is used for orientation and concept coverage only. The prose, examples, diagrams, code, and checks in this notebook are original.

**Chapter question:** What algebraic data is needed before geometry can be attached to a group?

This chapter supplies the algebraic raw material: subgroups, quotients, kernels, generators, relations, free groups, products, and extensions. For a visualization-first course, the central computational idea is that a presentation is a machine for producing identifications. Before relations are imposed, reduced words in a free group form a tree. After relations are imposed, formerly distinct words can land on the same vertex, and the Cayley graph records the quotient geometry.

The notebook uses the dihedral group as a finite laboratory because every relation can be checked exactly. Rotation and reflection generate a concrete symmetry group; the relators force a finite Cayley graph; the resulting graph is regular because every group element has the same generator moves available.

This notebook is standalone: it introduces the working objects, builds the relevant finite models, saves artifacts under `artifacts/chapter-02/`, and ends with sanity checks. When a finite graph window is used, treat it as a controlled model of the local or coarse pattern, not as a claim that the infinite group has been exhausted.

In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Geometric-Group-Theory-An-Introduction/part-01-groups/chapter-02-generating-groups/02-generating-groups.ipynb",
  "course_dir": "Geometric-Group-Theory-An-Introduction",
  "course_title": "Geometric Group Theory: An Introduction",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Geometric-Group-Theory-An-Introduction/part-01-groups/chapter-02-generating-groups/02-generating-groups.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Geometric-Group-Theory-An-Introduction/part-01-groups/chapter-02-generating-groups/02-generating-groups.ipynb",
  "notebook_title": "Chapter 02: Generating Groups",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/algebraic-geometry.txt",
  "runtime_profile": "algebraic_geometry"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

- A generating set is an alphabet for writing group elements.
- A free group is the space of reduced words before extra equations are imposed.
- A normal closure of relators is the bookkeeping device that says which words become invisible in the quotient.
- A presentation is useful geometrically because it predicts cycles in a Cayley graph.
- Product and extension constructions can be read as ways of assembling new metric graph models from old ones.

## Library routing

NetworkX renders the finite Cayley graph and relation pipeline; exact integer arithmetic checks the dihedral relations; Matplotlib records regularity and presentation structure; Plotly provides an interactive finite Cayley graph where vertices are group elements.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
for candidate in (HERE, *HERE.parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the Geometric Group Theory course root.")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

BOOK_ROOT

## Visual Storyboard

The overview visual puts two views side by side: the concrete Cayley graph of a dihedral group and the abstract pipeline from alphabet to quotient. The diagnostic visual checks that the Cayley graph is regular, reinforcing that generator moves are attached uniformly to every group element rather than to a preferred basepoint.

Each visual has a nearby computational check. The check is intentionally small enough to be readable: graph connectivity, relation identities, distance distortion bounds, hyperbolicity samples, component counts, boundary ratios, or model-distance sanity tests. The artifacts are saved before display so the notebook can be audited outside the live kernel.

In [ ]:
from utils.chapter_visuals import build_generating_groups_visuals

UNIT = "chapter-02"
outputs = build_generating_groups_visuals(UNIT)
outputs

## What To Inspect

In the Cayley graph, separate rotation vertices from reflection vertices and notice how the reflection generator jumps between the two rings. The relation check is the computational version of a presentation: powers and alternating products are evaluated in the group law, and the output must return to the identity exactly.

The useful habit is to ask which feature of the picture survives when the finite drawing is enlarged. Vertices at the edge of a rendered ball are artifacts of the cut window; branching, cycles, distance envelopes, shrinking ratios, and stable component counts are the features meant to carry mathematical information.

In [ ]:
from utils.artifacts import display_artifact

artifact_root = BOOK_ROOT / "artifacts" / "chapter-02"
display_artifact(artifact_root / "figures" / "generators-relations-and-dihedral-cayley.png", width=920)
display_artifact(artifact_root / "figures" / "dihedral-degree-regularity.png", width=920)

In [ ]:
from utils.artifacts import display_artifact, read_json

artifact_root = BOOK_ROOT / "artifacts" / "chapter-02"
display_artifact(artifact_root / "html" / "dihedral-cayley-graph.html", height=560)
checks = read_json(artifact_root / "checks" / "presentation-relation-checks.json")
checks

## Applied Lab

Applied lab: change the rotation order from 8 to another integer n in the helper call. Predict the order of the group and the two-ring geometry before looking at the image. This is a low-risk way to see how presentations control finite models and why changing a relator changes global graph size.

A good extension of this lab should add one new parameter, state the predicted invariant before running code, and save a check JSON next to any new figure. That pattern keeps experimentation tied to proof-relevant evidence rather than to attractive but untested pictures.

## Takeaways

        - Generators create words; relations create identifications.
- Finite presentations are geometric instructions for Cayley graph cycles.
- Exact relation checks keep the computational model honest.

        The final cell below re-reads the saved artifact manifest and the chapter-specific check file. It is deliberately redundant: rerunning the notebook should both rebuild the visuals and verify that the saved course assets remain present, nonempty, and mathematically consistent.

In [ ]:
from utils.artifacts import assert_artifact, read_json

artifact_root = BOOK_ROOT / "artifacts" / "chapter-02"
final = read_json(artifact_root / "checks" / "final-sanity.json")
checks = read_json(artifact_root / "checks" / "presentation-relation-checks.json")

for row in final["artifacts"]:
    path = BOOK_ROOT / row["path"]
    assert_artifact(
        path,
        min_bytes=1024 if path.suffix.lower() == ".html" else 512,
        nonblank_image=path.suffix.lower() == ".png",
    )

assert checks["r_to_8_identity"] and checks["s_to_2_identity"] and checks["sr_to_2_identity"]
final